# P0.1 completion - checkpoint verification + locked-test evaluation

Everything P0.1 needs in one run:

1. **Verify the checkpoint contract** - loads safely, class order matches the frozen manifest, parameter count, recorded validation score, SHA-256 fingerprint.
2. **Re-measure validation** so the claimed 99.54% accuracy / 99.37% macro-F1 is independently reproduced rather than trusted.
3. **Score the locked test split once** - the artifact that never existed: per-class report, 38x38 confusion matrix, bootstrap 95% CIs, calibration.
4. **Error analysis** - weakest classes, top off-diagonal confusions, and the cross-crop healthy-class check the audit flagged.
5. **Write `P0.1_VERIFICATION.md`** with every number filled in, ready to commit as evidence.

It trains nothing and never uses the test split for model selection.

**Attach in Kaggle**

| Attachment | Gives |
|---|---|
| PlantVillage dataset | `data/manifests`, `data/raw/PlantVillage` |
| checkpoint dataset | `efficientnet_b0_best.pt` |

Set `CHECKPOINT_PATH` and `DATASET_ROOT` in cell 1. Nothing else needs editing.

Outputs land in `/kaggle/working/p01_verification/` and are zipped for download.

### Run order matters

| Cell | Does | Why the order matters |
|---|---|---|
| 1 | sets paths, creates the folder tree | everything else reads these paths |
| 2 | writes the package and the evaluator, then imports them to prove it worked | this is the only cell that can create the code, so it cannot be skipped |
| 3 | checkpoint contract check | imports `plant_disease.models` |
| 4 | runs the evaluation | needs 1 and 3 |
| 5 | headline results | needs 4 |
| 6 | error analysis | needs 4 |
| 7 | evidence record + zip | needs 4 and 6 |

Cell 2 prints `package import OK` when the code is in place. If the next cell ever fails with `unknown location`, cell 2 has not run.

## 1. Paths

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

import numpy as np
import pandas as pd
import torch

# ---- paths ------------------------------------------------------------------
# Kaggle mounts a dataset at /kaggle/input/<slug>. The slug and any intermediate
# folder depend on how the dataset was uploaded, so these are defaults that get
# corrected by discovery below. Override either one if discovery picks wrong.
DATASET_ROOT = Path("/kaggle/input/datasets/shubhamkumar12864/plantvillage")
CHECKPOINT_PATH = Path("/kaggle/input/efficientnet/efficientnet_b0_best.pt")


KAGGLE_INPUT = Path("/kaggle/input")   # monkeypatched by the local test


def autodiscover_dataset_root(preferred: Path) -> Path:
    """Find the attached dataset that holds data/manifests."""
    if (preferred / "data" / "manifests" / "plantvillage_test.csv").is_file():
        return preferred
    base = KAGGLE_INPUT
    if not base.is_dir():
        return preferred
    for manifests in sorted(base.rglob("manifests")):
        if (manifests / "plantvillage_test.csv").is_file():
            return manifests.parent.parent
    return preferred


def autodiscover_checkpoint(preferred: Path) -> Path:
    """Find the attached checkpoint, preferring the requested one."""
    if preferred.is_file():
        return preferred
    base = KAGGLE_INPUT
    if base.is_dir():
        hits = sorted(base.rglob("*.pt"))
        exact = [h for h in hits if h.name == preferred.name]
        if exact:
            return exact[0]
        if len(hits) == 1:
            return hits[0]
    return preferred


_resolved_root = autodiscover_dataset_root(DATASET_ROOT)
if _resolved_root != DATASET_ROOT:
    print(f"dataset root corrected: {DATASET_ROOT} -> {_resolved_root}")
DATASET_ROOT = _resolved_root

_resolved_ckpt = autodiscover_checkpoint(CHECKPOINT_PATH)
if _resolved_ckpt != CHECKPOINT_PATH:
    print(f"checkpoint corrected: {CHECKPOINT_PATH} -> {_resolved_ckpt}")
CHECKPOINT_PATH = _resolved_ckpt

# ---- run settings -----------------------------------------------------------
RUN_VALIDATION = True    # re-measure validation to reproduce the claimed score
RUN_TEST = True          # locked test split, scored once
BOOTSTRAP = 2000         # resamples for the confidence intervals
BATCH_SIZE = 128
WORKERS = 4

WORKDIR = Path("/kaggle/working")
SCRIPT = WORKDIR / "evaluate_locked_test.py"
OUT_ROOT = WORKDIR / "p01_verification"

# The %%writefile cells below need this tree to exist first.
(WORKDIR / "src" / "plant_disease" / "models").mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Paths ready. Next: run the writefile cells, then the checkpoint check.")
print(f"  dataset    : {DATASET_ROOT}")
print(f"  checkpoint : {CHECKPOINT_PATH}")

## 2. Write and import the evaluation code

One cell writes the package and the evaluator, then imports the package to prove it works. There is no ordering to get wrong.

In [ ]:
# Writes the package and the evaluator, then imports them. One cell, so
# the code always exists before anything tries to use it.
FILES = {
    '/kaggle/working/src/plant_disease/__init__.py': '"""Plant leaf disease classification utilities."""\n\n__version__ = "0.1.0"\n',
    '/kaggle/working/src/plant_disease/calibration.py': '"""Confidence calibration diagnostics.\n\nThe project plan lists ``calibration`` as a required metric but nothing computed\nit. This module provides the standard set so the report can state a measured\nvalue instead of claiming calibrated confidence without evidence.\n\nDefinitions used here\n---------------------\nECE (expected calibration error)\n    Bin predictions by top-1 confidence, then average |accuracy - confidence|\n    weighted by bin population. Lower is better; 0 is perfect calibration.\nMCE (maximum calibration error)\n    The worst single bin. Useful because ECE can hide one badly calibrated bin.\nBrier score\n    Mean squared error between the predicted probability vector and the one-hot\n    label. Lower is better. Rewards both calibration and sharpness.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\n\n\n@dataclass\nclass ReliabilityBin:\n    lower: float\n    upper: float\n    count: int\n    mean_confidence: float\n    accuracy: float\n\n\ndef reliability_bins(probabilities: np.ndarray, y_true: np.ndarray, n_bins: int = 15) -> list[ReliabilityBin]:\n    """Bin predictions by top-1 confidence for a reliability diagram."""\n    confidence = probabilities.max(axis=1)\n    correct = probabilities.argmax(axis=1) == y_true\n    edges = np.linspace(0.0, 1.0, n_bins + 1)\n    bins: list[ReliabilityBin] = []\n    for low, high in zip(edges[:-1], edges[1:]):\n        mask = (confidence > low) & (confidence <= high)\n        count = int(mask.sum())\n        bins.append(\n            ReliabilityBin(\n                lower=float(low),\n                upper=float(high),\n                count=count,\n                mean_confidence=float(confidence[mask].mean()) if count else 0.0,\n                accuracy=float(correct[mask].mean()) if count else 0.0,\n            )\n        )\n    return bins\n\n\ndef expected_calibration_error(probabilities: np.ndarray, y_true: np.ndarray, n_bins: int = 15) -> float:\n    total = len(y_true)\n    if total == 0:\n        raise ValueError("Cannot compute calibration on an empty set")\n    return sum(bin_.count / total * abs(bin_.accuracy - bin_.mean_confidence) for bin_ in reliability_bins(probabilities, y_true, n_bins))\n\n\ndef maximum_calibration_error(probabilities: np.ndarray, y_true: np.ndarray, n_bins: int = 15) -> float:\n    gaps = [abs(b.accuracy - b.mean_confidence) for b in reliability_bins(probabilities, y_true, n_bins) if b.count]\n    return float(max(gaps)) if gaps else 0.0\n\n\ndef brier_score(probabilities: np.ndarray, y_true: np.ndarray) -> float:\n    one_hot = np.zeros_like(probabilities)\n    one_hot[np.arange(len(y_true)), y_true] = 1.0\n    return float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))\n\n\ndef selective_risk_coverage(probabilities: np.ndarray, y_true: np.ndarray, thresholds=None) -> list[dict]:\n    """Accuracy retained when predictions below a confidence threshold abstain.\n\n    This is the straightforward version of the ``selective_risk_coverage`` metric\n    in ``configs/data_contract.yaml``: for each threshold, how much of the set is\n    answered and how accurate those answers are.\n    """\n    if thresholds is None:\n        thresholds = [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]\n    confidence = probabilities.max(axis=1)\n    correct = probabilities.argmax(axis=1) == y_true\n    rows = []\n    for threshold in thresholds:\n        answered = confidence >= threshold\n        coverage = float(answered.mean())\n        accuracy = float(correct[answered].mean()) if answered.any() else float("nan")\n        rows.append({"threshold": float(threshold), "coverage": coverage, "selective_accuracy": accuracy})\n    return rows\n\n\ndef summarise(probabilities: np.ndarray, y_true: np.ndarray, n_bins: int = 15) -> dict:\n    """All calibration numbers in one dict, ready for JSON."""\n    confidence = probabilities.max(axis=1)\n    return {\n        "ece": expected_calibration_error(probabilities, y_true, n_bins),\n        "mce": maximum_calibration_error(probabilities, y_true, n_bins),\n        "brier": brier_score(probabilities, y_true),\n        "mean_confidence": float(confidence.mean()),\n        "median_confidence": float(np.median(confidence)),\n        "fraction_below_0_5": float((confidence < 0.5).mean()),\n        "fraction_below_0_8": float((confidence < 0.8).mean()),\n        "n_bins": n_bins,\n    }\n\n\ndef to_markdown_table(bins: list[ReliabilityBin]) -> str:\n    """Reliability bins as a markdown table for the report."""\n    lines = ["| Bin | Count | Mean confidence | Accuracy | Gap |", "|---|---:|---:|---:|---:|"]\n    for bin_ in bins:\n        lines.append(\n            f"| {bin_.lower:.2f}-{bin_.upper:.2f} | {bin_.count} | "\n            f"{bin_.mean_confidence:.3f} | {bin_.accuracy:.3f} | {abs(bin_.accuracy - bin_.mean_confidence):.3f} |"\n        )\n    return "\\n".join(lines)\n',
    '/kaggle/working/src/plant_disease/data.py': 'from pathlib import Path\n\nimport pandas as pd\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom torchvision import transforms\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\nSMALLCNN_MEAN = (0.5, 0.5, 0.5)\nSMALLCNN_STD = (0.5, 0.5, 0.5)\n\n\ndef resolve_image_root(dataset_root: Path, frame: pd.DataFrame, sample: int = 25) -> Path:\n    """Find where a manifest\'s images actually live.\n\n    PlantVillage is materialised with two roots: the development pool (train and\n    validation rows) sits under ``data/raw/PlantVillage/train``, while the\n    published test rows sit under ``data/raw/PlantVillage/test``. Hard-coding\n    either one breaks the other split, so resolve it from the manifest itself.\n    """\n    base = Path(dataset_root) / "data" / "raw" / "PlantVillage"\n    if not base.is_dir():\n        raise FileNotFoundError(f"PlantVillage images not found under {base}")\n\n    relative = frame["relative_path"].head(sample).tolist()\n    if not relative:\n        raise ValueError("Cannot resolve an image root for an empty manifest")\n\n    candidates = [base / "train", base / "test"]\n    for candidate in candidates:\n        if candidate.is_dir() and all((candidate / rel).is_file() for rel in relative):\n            return candidate\n\n    scored = sorted(\n        ((sum((candidate / rel).is_file() for rel in relative), candidate) for candidate in candidates),\n        key=lambda item: item[0],\n        reverse=True,\n    )\n    if scored[0][0] > 0:\n        return scored[0][1]\n    raise FileNotFoundError(\n        f"None of {[str(c) for c in candidates]} contains the manifest\'s images. "\n        f"First expected file: {candidates[0] / relative[0]}"\n    )\n\n\ndef load_split(dataset_root: Path, split: str):\n    """Load train/validation rows from the development manifest only."""\n    if split not in {"train", "validation"}:\n        raise ValueError("This comparison tool accepts train or validation. Use the final-evaluation notebook for test data.")\n    root = Path(dataset_root)\n    manifest = root / "data" / "manifests" / "plantvillage_development.csv"\n    if not manifest.is_file():\n        raise FileNotFoundError(f"Missing development manifest: {manifest}")\n    frame = pd.read_csv(manifest)\n    required = {"split", "relative_path", "label"}\n    if not required.issubset(frame.columns):\n        raise ValueError(f"Manifest missing columns: {sorted(required - set(frame.columns))}")\n    frame = frame.loc[frame["split"] == split].reset_index(drop=True)\n    classes = sorted(frame["label"].dropna().unique().tolist())\n    if frame.empty or not classes:\n        raise ValueError(f"No rows found for split={split!r} in {manifest}")\n    return frame, classes\n\n\nclass PlantVillageImages(Dataset):\n    def __init__(self, dataset_root: Path, frame: pd.DataFrame, classes: list[str], model_name: str, image_size=224):\n        self.image_root = Path(dataset_root) / "data" / "raw" / "PlantVillage" / "train"\n        self.rows = frame.reset_index(drop=True)\n        self.label_to_index = {label: index for index, label in enumerate(classes)}\n        self.model_name = model_name\n        mean, std = (SMALLCNN_MEAN, SMALLCNN_STD) if model_name == "smallcnn" else (IMAGENET_MEAN, IMAGENET_STD)\n        self.transform = transforms.Compose(\n            [transforms.Resize((image_size, image_size)), transforms.ToTensor(), transforms.Normalize(mean, std)]\n        )\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows.iloc[index]\n        path = self.image_root / row["relative_path"]\n        with Image.open(path) as image:\n            tensor = self.transform(image.convert("RGB"))\n        return tensor, self.label_to_index[row["label"]]\n',
    '/kaggle/working/src/plant_disease/models/__init__.py': '"""Model definitions used by the PlantVillage project."""\n\nfrom torch import nn\n\nfrom .efficientnet_b0 import build_efficientnet_b0\nfrom .resnet18 import build_resnet18\nfrom .smallcnn import SmallCNN\n\nMODEL_NAMES = ("smallcnn", "resnet18", "efficientnet_b0")\n\n\ndef build_model(name: str, num_classes: int, pretrained: bool = False) -> nn.Module:\n    """Build one supported architecture with a project-sized output layer."""\n    if name == "smallcnn":\n        return SmallCNN(num_classes)\n    if name == "resnet18":\n        return build_resnet18(num_classes, pretrained=pretrained)\n    if name == "efficientnet_b0":\n        return build_efficientnet_b0(num_classes, pretrained=pretrained)\n    raise ValueError(f"Unknown model {name!r}. Choose from: {\', \'.join(MODEL_NAMES)}")\n',
    '/kaggle/working/src/plant_disease/models/efficientnet_b0.py': 'from torch import nn\nfrom torchvision import models\n\n\ndef build_efficientnet_b0(num_classes: int, pretrained: bool = False) -> nn.Module:\n    weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None\n    model = models.efficientnet_b0(weights=weights)\n    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)\n    return model\n',
    '/kaggle/working/src/plant_disease/models/resnet18.py': 'from torch import nn\nfrom torchvision import models\n\n\ndef build_resnet18(num_classes: int, pretrained: bool = False) -> nn.Module:\n    weights = models.ResNet18_Weights.DEFAULT if pretrained else None\n    model = models.resnet18(weights=weights)\n    model.fc = nn.Linear(model.fc.in_features, num_classes)\n    return model\n',
    '/kaggle/working/src/plant_disease/models/smallcnn.py': 'from torch import nn\n\n\nclass SmallCNN(nn.Module):\n    """Four-stage CNN used as the project baseline."""\n\n    def __init__(self, num_classes: int):\n        super().__init__()\n        blocks = []\n        input_channels = 3\n        for output_channels in (32, 64, 128, 256):\n            blocks.extend(\n                [\n                    nn.Conv2d(input_channels, output_channels, 3, padding=1, bias=False),\n                    nn.BatchNorm2d(output_channels),\n                    nn.ReLU(inplace=True),\n                    nn.MaxPool2d(2),\n                ]\n            )\n            input_channels = output_channels\n        self.features = nn.Sequential(*blocks)\n        self.head = nn.Sequential(\n            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.30), nn.Linear(256, num_classes)\n        )\n\n    def forward(self, images):\n        return self.head(self.features(images))\n',
    '/kaggle/working/evaluate_locked_test.py': '#!/usr/bin/env python3\n"""Score any saved checkpoint on any frozen PlantVillage split, locally.\n\nWhy this exists\n---------------\nThe audit found no local test-evaluation path: ``src/plant_disease/data.py``\nhard-codes the training image root and ``evaluate.py`` refuses any split other\nthan validation, so test scores existed only inside Notebook 2. That is why the\nREADME\'s test numbers could not be checked in the repository.\n\nThis script evaluates ``train``, ``validation`` or ``test`` from the frozen\nmanifests and writes everything a defensible results section needs:\n\n* headline metrics (accuracy, balanced accuracy, macro P/R/F1, weighted F1)\n* per-class precision/recall/F1/support\n* the full confusion matrix\n* image-level bootstrap confidence intervals\n* calibration diagnostics (ECE, MCE, Brier) and selective risk/coverage\n\nUsage\n-----\n    python scripts/evaluate_locked_test.py \\\n        --checkpoint checkpoints/efficientnet_b0/efficientnet_b0_best.pt \\\n        --dataset-root data \\\n        --split test \\\n        --output-dir reports/model_comparison/locked_test_efficientnet_b0\n\nThe output is JSON plus CSV so both the report and the guide form can cite the\nsame numbers.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.metrics import (\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_fscore_support,\n    top_k_accuracy_score,\n)\nfrom torch.utils.data import DataLoader, Dataset\nfrom tqdm.auto import tqdm\n\n_SCRIPT_PATH = Path(__file__).resolve()\n# The package can sit one level up (repository layout: scripts/ + src/) or beside\n# this file (Kaggle layout: /kaggle/working/evaluate_locked_test.py with\n# /kaggle/working/src/plant_disease). Guessing parents[1] silently produced\n# "/kaggle/src", which does not exist and broke the subprocess import.\nROOT = _SCRIPT_PATH.parents[1] if (_SCRIPT_PATH.parents[1] / "src").is_dir() else _SCRIPT_PATH.parent\nsys.path.insert(0, str(ROOT / "src"))\n\nfrom plant_disease.data import resolve_image_root  # noqa: E402\nfrom plant_disease.calibration import (  # noqa: E402\n    reliability_bins,\n    selective_risk_coverage,\n    summarise as calibration_summary,\n    to_markdown_table,\n)\nfrom plant_disease.models import build_model  # noqa: E402\n\nMANIFESTS = {\n    "train": "plantvillage_train.csv",\n    "validation": "plantvillage_validation.csv",\n    "test": "plantvillage_test.csv",\n}\nDEV_MANIFEST = "plantvillage_development.csv"\n\n\nclass ManifestImages(Dataset):\n    """Images addressed by manifest row, resized to the model\'s training size."""\n\n    def __init__(self, rows: pd.DataFrame, image_root: Path, class_to_index: dict, image_size: int):\n        self.rows = rows.reset_index(drop=True)\n        self.image_root = image_root\n        self.class_to_index = class_to_index\n        self.image_size = image_size\n        self.missing = [\n            str(image_root / rel) for rel in self.rows["relative_path"]\n            if not (image_root / rel).is_file()\n        ]\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def __getitem__(self, index: int):\n        from PIL import Image\n\n        row = self.rows.iloc[index]\n        with Image.open(self.image_root / row["relative_path"]) as image:\n            array = image.convert("RGB").resize(\n                (self.image_size, self.image_size), Image.Resampling.BILINEAR\n            )\n        tensor = torch.from_numpy(np.asarray(array, dtype=np.uint8).copy())\n        return tensor, self.class_to_index[row["label"]]\n\n\ndef load_manifest(dataset_root: Path, split: str) -> pd.DataFrame:\n    manifest_dir = dataset_root / "data" / "manifests"\n    if split in ("train", "validation"):\n        path = manifest_dir / DEV_MANIFEST\n        frame = pd.read_csv(path)\n        return frame.loc[frame["split"] == split].reset_index(drop=True)\n    path = manifest_dir / MANIFESTS[split]\n    if not path.is_file():\n        raise FileNotFoundError(f"Missing manifest for split {split!r}: {path}")\n    return pd.read_csv(path)\n\n\ndef safe_load(path: Path, map_location="cpu"):\n    try:\n        return torch.load(path, map_location=map_location, weights_only=True)\n    except Exception as exc:\n        print(f"  weights_only=True failed ({exc}); retrying in compatibility mode", flush=True)\n        return torch.load(path, map_location=map_location, weights_only=False)\n\n\ndef clean_state_dict(state):\n    if state and all(str(key).startswith("module.") for key in state):\n        return {key[7:]: value for key, value in state.items()}\n    return state\n\n\ndef bootstrap_ci(y_true: np.ndarray, y_pred: np.ndarray, n_resamples: int, seed: int = 42) -> dict:\n    """Image-level bootstrap over metric differences. Percentile intervals."""\n    rng = np.random.default_rng(seed)\n    n = len(y_true)\n    if n_resamples <= 0 or n == 0:\n        return {}\n    point = {\n        "accuracy": float((y_true == y_pred).mean()),\n        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),\n        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),\n    }\n    collected = {key: [] for key in point}\n    for _ in range(n_resamples):\n        index = rng.integers(0, n, n)\n        collected["accuracy"].append(float((y_true[index] == y_pred[index]).mean()))\n        collected["balanced_accuracy"].append(float(balanced_accuracy_score(y_true[index], y_pred[index])))\n        collected["macro_f1"].append(\n            float(f1_score(y_true[index], y_pred[index], average="macro", zero_division=0))\n        )\n    out = {}\n    for key, values in collected.items():\n        low, high = np.percentile(values, [2.5, 97.5])\n        out[key] = {"point": point[key], "ci95_low": float(low), "ci95_high": float(high)}\n    out["resamples"] = n_resamples\n    return out\n\n\n@torch.no_grad()\ndef run_inference(model, dataset, device, mean, std, batch_size, workers, use_cuda):\n    loader = DataLoader(\n        dataset, batch_size=batch_size, shuffle=False,\n        num_workers=workers, pin_memory=use_cuda,\n        persistent_workers=workers > 0,\n    )\n    y_true, y_pred, probability_rows = [], [], []\n    started = time.perf_counter()\n    for images, labels in tqdm(loader, desc="inference", unit="batch"):\n        x = images.to(device=device, dtype=torch.float32, non_blocking=True)\n        x = (x.permute(0, 3, 1, 2).div_(255.0) - torch.tensor(mean, device=device).view(1, 3, 1, 1)) / (\n            torch.tensor(std, device=device).view(1, 3, 1, 1)\n        )\n        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=use_cuda):\n            logits = model(x)\n        probabilities = F.softmax(logits.float(), dim=1).cpu().numpy()\n        y_true.extend(labels.numpy().tolist())\n        y_pred.extend(probabilities.argmax(axis=1).tolist())\n        probability_rows.append(probabilities)\n    elapsed = time.perf_counter() - started\n    return np.asarray(y_true), np.asarray(y_pred), np.concatenate(probability_rows, axis=0), elapsed\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--checkpoint", type=Path, required=True)\n    parser.add_argument("--dataset-root", type=Path, default=ROOT,\n                        help="Repository root containing data/manifests and data/raw/PlantVillage")\n    parser.add_argument("--split", default="test", choices=("train", "validation", "test"))\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--batch-size", type=int, default=128)\n    parser.add_argument("--workers", type=int, default=4)\n    parser.add_argument("--bootstrap", type=int, default=2000, help="bootstrap resamples; 0 disables")\n    parser.add_argument("--limit", type=int, default=0, help="score only the first N rows (smoke checks)")\n    parser.add_argument("--device", default="auto", choices=("auto", "cuda", "cpu"))\n    args = parser.parse_args()\n\n    checkpoint = safe_load(args.checkpoint)\n    classes = checkpoint.get("classes")\n    if not classes:\n        raise ValueError("Checkpoint has no ordered class list; refusing to guess a class order.")\n    architecture = checkpoint.get("architecture")\n    if architecture not in ("smallcnn", "resnet18", "efficientnet_b0"):\n        raise ValueError(f"Unsupported or missing architecture in checkpoint: {architecture!r}")\n\n    image_size = int(checkpoint.get("image_size", 224))\n    mean = tuple(checkpoint.get("normalization_mean", (0.485, 0.456, 0.406)))\n    std = tuple(checkpoint.get("normalization_std", (0.229, 0.224, 0.225)))\n\n    frame = load_manifest(args.dataset_root, args.split)\n    if args.limit:\n        frame = frame.head(args.limit).reset_index(drop=True)\n    unseen = sorted(set(frame["label"]) - set(classes))\n    if unseen:\n        raise ValueError(f"Manifest has labels the checkpoint cannot predict: {unseen[:5]}")\n\n    use_cuda = args.device == "cuda" or (args.device == "auto" and torch.cuda.is_available())\n    device = torch.device("cuda" if use_cuda else "cpu")\n\n    model = build_model(architecture, len(classes), pretrained=False)\n    model.load_state_dict(clean_state_dict(checkpoint.get("model_state_dict", checkpoint.get("state_dict"))))\n    model.to(device).eval()\n\n    image_root = resolve_image_root(args.dataset_root, frame)\n    dataset = ManifestImages(frame, image_root, {label: i for i, label in enumerate(classes)}, image_size)\n    print(f"image root  : {image_root}")\n    if dataset.missing:\n        raise FileNotFoundError(f"{len(dataset.missing)} images missing; first: {dataset.missing[0]}")\n\n    y_true, y_pred, probabilities, elapsed = run_inference(\n        model, dataset, device, mean, std, args.batch_size, args.workers, use_cuda\n    )\n\n    precision, recall, f1, support = precision_recall_fscore_support(\n        y_true, y_pred, labels=np.arange(len(classes)), zero_division=0\n    )\n    per_class = pd.DataFrame(\n        {"class": classes, "precision": precision, "recall": recall, "f1": f1, "support": support}\n    )\n    labels_index = np.arange(len(classes))\n    metrics = {\n        "model": architecture,\n        "split": args.split,\n        "dataset": "PlantVillage",\n        "images": int(len(y_true)),\n        "accuracy": float((y_true == y_pred).mean()),\n        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),\n        "macro_precision": float(precision.mean()),\n        "macro_recall": float(recall.mean()),\n        "macro_f1": float(f1.mean()),\n        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),\n        "top3_accuracy": float(top_k_accuracy_score(y_true, probabilities, k=3, labels=labels_index)),\n        "classes_with_zero_support": [c for c, s in zip(classes, support) if s == 0],\n        "checkpoint": str(args.checkpoint),\n        "checkpoint_dataset_fingerprint": checkpoint.get("dataset_fingerprint"),\n        "inference_seconds": round(elapsed, 2),\n        "inference_ms_per_image": round(1000.0 * elapsed / max(len(y_true), 1), 3),\n        "device": str(device),\n        **calibration_summary(probabilities, y_true),\n    }\n    metrics["bootstrap"] = bootstrap_ci(y_true, y_pred, args.bootstrap)\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    (args.output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")\n    per_class.sort_values("f1").to_csv(args.output_dir / "classification_report.csv", index=False)\n    pd.DataFrame(\n        confusion_matrix(y_true, y_pred, labels=labels_index), index=classes, columns=classes\n    ).to_csv(args.output_dir / "confusion_matrix.csv")\n    pd.DataFrame(selective_risk_coverage(probabilities, y_true)).to_csv(\n        args.output_dir / "selective_risk_coverage.csv", index=False\n    )\n    bins = reliability_bins(probabilities, y_true)\n    pd.DataFrame(\n        [{"lower": b.lower, "upper": b.upper, "count": b.count,\n          "mean_confidence": b.mean_confidence, "accuracy": b.accuracy} for b in bins]\n    ).to_csv(args.output_dir / "reliability_bins.csv", index=False)\n    (args.output_dir / "reliability.md").write_text(to_markdown_table(bins) + "\\n", encoding="utf-8")\n\n    print(json.dumps({k: metrics[k] for k in (\n        "model", "split", "images", "accuracy", "balanced_accuracy", "macro_f1", "ece", "brier"\n    )}, indent=2))\n    print(f"\\nWrote results to {args.output_dir}")\n    print("Weakest five classes by F1:")\n    print(per_class.sort_values("f1").head(5).to_string(index=False))\n\n\nif __name__ == "__main__":\n    main()\n',
}

for _path, _content in FILES.items():
    _target = Path(_path)
    _target.parent.mkdir(parents=True, exist_ok=True)
    _target.write_text(_content, encoding='utf-8')
    print(f'wrote {_target.name:24s} {len(_content):6d} bytes')
print(f'{len(FILES)} files written to', Path('/kaggle/working/src'))

# Prove the package imports before anything else uses it.
if str(WORKDIR / 'src') not in sys.path:
    sys.path.insert(0, str(WORKDIR / 'src'))
import importlib
import plant_disease.models as _models
importlib.reload(_models)
if not hasattr(_models, 'build_model'):
    raise ImportError('plant_disease.models imported but exposes no build_model; files may be stale')
print('package import OK')

## 3. Check the checkpoint

In [ ]:
# This cell needs the plant_disease package, which the writefile cells above create.
_sys_path_added = str(WORKDIR / "src")
if _sys_path_added not in sys.path:
    sys.path.insert(0, _sys_path_added)

try:
    from plant_disease.models import build_model
except ImportError as exc:
    raise ImportError(
        "The plant_disease package is not on disk yet. Run the %%writefile cells above "
        "(they create /kaggle/working/src/plant_disease). Original error: " + str(exc)
    )

# ---- preflight: inputs ------------------------------------------------------
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset not attached at {DATASET_ROOT}")
manifest_dir = DATASET_ROOT / "data" / "manifests"
for required in ("plantvillage_development.csv", "plantvillage_test.csv"):
    if not (manifest_dir / required).is_file():
        raise FileNotFoundError(f"{required} missing from {manifest_dir}")
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not attached at {CHECKPOINT_PATH}")

# ---- preflight: checkpoint contract ----------------------------------------
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
ARCH = checkpoint["architecture"]
classes = checkpoint["classes"]
state = checkpoint.get("model_state_dict", checkpoint.get("state_dict"))
if all(str(k).startswith("module.") for k in state):
    state = {k[7:]: v for k, v in state.items()}

dev = pd.read_csv(manifest_dir / "plantvillage_development.csv")
manifest_classes = sorted(dev.loc[dev.split == "train", "label"].unique().tolist())
if classes != manifest_classes:
    raise ValueError(
        "Checkpoint class order does not match the development manifest. "
        f"first difference: {next(((i, a, b) for i, (a, b) in enumerate(zip(classes, manifest_classes)) if a != b), None)}"
    )

model = build_model(ARCH, len(classes), pretrained=False)
model.load_state_dict(state, strict=True)
parameters = sum(p.numel() for p in model.parameters())
del model

sha256 = hashlib.sha256()
with CHECKPOINT_PATH.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1 << 20), b""):
        sha256.update(chunk)

print("CHECKPOINT OK")
print(f"  path        : {CHECKPOINT_PATH}")
print(f"  sha256      : {sha256.hexdigest()}")
print(f"  size        : {CHECKPOINT_PATH.stat().st_size / 1024**2:.2f} MB")
print(f"  architecture: {ARCH}")
print(f"  parameters  : {parameters:,}")
print(f"  classes     : {len(classes)} (order matches the development manifest)")
print(f"  image size  : {checkpoint.get('image_size')}")
print(f"  epoch       : {checkpoint.get('epoch')}")
print(f"  val_macro_f1 recorded: {checkpoint.get('val_macro_f1')}")
print(f"  fingerprint : {checkpoint.get('dataset_fingerprint', 'absent (pilot checkpoint)')}")
print(f"  device      : {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 4. Run the evaluation

In [ ]:
env = os.environ.copy()
env.update({"PYTHONUNBUFFERED": "1", "OMP_NUM_THREADS": "4"})

splits = []
if RUN_VALIDATION:
    splits.append("validation")
if RUN_TEST:
    splits.append("test")

outputs = {}
for split in splits:
    out = OUT_ROOT / split
    command = [
        sys.executable, str(SCRIPT),
        "--checkpoint", str(CHECKPOINT_PATH),
        "--dataset-root", str(DATASET_ROOT),
        "--split", split,
        "--output-dir", str(out),
        "--batch-size", str(BATCH_SIZE),
        "--workers", str(WORKERS),
        "--bootstrap", str(BOOTSTRAP),
        "--device", "auto",
    ]
    print(f"\n=== {split} ===", flush=True)
    print("Running:", " ".join(command[-8:]), flush=True)
    result = subprocess.run(command, env=env, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"Evaluation failed for {split} with code {result.returncode}. Read the log above.")
    outputs[split] = out

print("\nAll requested splits evaluated:", sorted(outputs))

## Headline results

In [ ]:
metrics = {split: json.loads((out / "metrics.json").read_text(encoding="utf-8"))
           for split, out in outputs.items()}

rows = []
for split, m in metrics.items():
    boot = m.get("bootstrap") or {}
    acc = boot.get("accuracy", {})
    f1 = boot.get("macro_f1", {})
    rows.append({
        "split": split,
        "images": m["images"],
        "accuracy": round(m["accuracy"], 5),
        "acc_ci95": f"[{acc.get('ci95_low', float('nan')):.4f}, {acc.get('ci95_high', float('nan')):.4f}]" if acc else "-",
        "balanced_accuracy": round(m["balanced_accuracy"], 5),
        "macro_precision": round(m["macro_precision"], 5),
        "macro_recall": round(m["macro_recall"], 5),
        "macro_f1": round(m["macro_f1"], 5),
        "macro_f1_ci95": f"[{f1.get('ci95_low', float('nan')):.4f}, {f1.get('ci95_high', float('nan')):.4f}]" if f1 else "-",
        "top3": round(m["top3_accuracy"], 5),
        "ece": round(m["ece"], 4),
        "mce": round(m["mce"], 4),
        "brier": round(m["brier"], 4),
        "ms_per_image": m["inference_ms_per_image"],
    })
headline = pd.DataFrame(rows)
display(headline)

print("Recorded values inside the checkpoint file:")
print(f"  val_macro_f1 = {checkpoint.get('val_macro_f1')}")
if "validation" in metrics:
    measured = metrics["validation"]["macro_f1"]
    recorded = checkpoint.get("val_macro_f1")
    delta = abs(measured - recorded) if recorded is not None else float("nan")
    verdict = "MATCH" if recorded is not None and delta < 1e-3 else "DIFFERS - investigate"
    print(f"  measured  val_macro_f1 = {measured:.6f}")
    print(f"  |difference| = {delta:.6f} -> {verdict}")
    print("  (a small gap is expected between GPU FP16 autocast training-time validation "
          "and CPU FP32 re-evaluation; a large gap is not)")

## Error analysis

In [ ]:
confusions = {}
for split, out in outputs.items():
    matrix_path = out / "confusion_matrix.csv"
    if not matrix_path.is_file():
        continue
    matrix = pd.read_csv(matrix_path, index_col=0)
    off = matrix.where(~np.eye(len(matrix), dtype=bool)).stack().dropna().sort_values(ascending=False)
    top = off.head(10).reset_index()
    top.columns = ["true_class", "predicted_class", "count"]
    # stack() upcasts to float because of the masked diagonal; counts are ints.
    top["count"] = top["count"].astype(int)
    top = top.loc[top["count"] > 0].reset_index(drop=True)
    confusions[split] = top
    print(f"\nTop off-diagonal confusions - {split}")
    if len(top):
        display(top)
    else:
        print("  none - every prediction was correct")

    report = pd.read_csv(out / "classification_report.csv").sort_values("f1")
    print(f"Weakest 8 classes by F1 - {split}")
    display(report.head(8))
    thin = report.loc[report.support <= 30].sort_values("support")
    if len(thin):
        print(f"Classes with <= 30 images of support - {split} (metrics on these are unstable)")
        display(thin)

    # Cross-crop healthy-class check the audit called out: Soybean healthy was
    # predicted as Pepper bell healthy in 20 of 32 errors. If that pattern is
    # still present it is evidence of background/appearance shortcuts.
    healthy = [c for c in matrix.index if "healthy" in c.lower()]
    if healthy:
        cross = matrix.loc[healthy, healthy]
        worst = (
            cross.where(~np.eye(len(cross), dtype=bool)).stack().dropna()
            .astype(int).sort_values(ascending=False).head(5)
        )
        worst = worst.loc[worst > 0]
        if len(worst):
            print(f"Healthy -> different-crop healthy confusions - {split}")
            print(worst.to_string())
        else:
            print(f"No cross-crop healthy confusions in {split}.")

## Evidence record and download bundle

In [ ]:
lines = [
    "# P0.1 verification record",
    "",
    f"Generated: {pd.Timestamp.now(tz='UTC').strftime('%Y-%m-%d %H:%M UTC')}",
    "",
    "## Checkpoint",
    "",
    "| Field | Value |",
    "|---|---|",
    f"| File | `{CHECKPOINT_PATH.name}` |",
    f"| SHA-256 | `{sha256.hexdigest()}` |",
    f"| Size | {CHECKPOINT_PATH.stat().st_size / 1024**2:.2f} MB |",
    f"| Architecture | {ARCH} |",
    f"| Parameters | {parameters:,} |",
    f"| Classes | {len(classes)} (order matches development manifest) |",
    f"| Image size | {checkpoint.get('image_size')} |",
    f"| Epoch | {checkpoint.get('epoch')} |",
    f"| Recorded val macro-F1 | {checkpoint.get('val_macro_f1')} |",
    f"| Dataset fingerprint | {checkpoint.get('dataset_fingerprint') or 'absent (pilot checkpoint)'} |",
    f"| Strict state-dict load | passed |",
    "",
    "## Measured performance",
    "",
]
for split, m in metrics.items():
    boot = m.get("bootstrap") or {}
    acc_ci = boot.get("accuracy", {})
    lines += [
        f"### {split} ({m['images']:,} images)",
        "",
        "| Metric | Value |",
        "|---|---|",
        f"| Accuracy | {m['accuracy']:.5f} |",
    ]
    if acc_ci:
        lines.append(f"| Accuracy 95% CI | [{acc_ci['ci95_low']:.5f}, {acc_ci['ci95_high']:.5f}] |")
    lines += [
        f"| Balanced accuracy | {m['balanced_accuracy']:.5f} |",
        f"| Macro precision | {m['macro_precision']:.5f} |",
        f"| Macro recall | {m['macro_recall']:.5f} |",
        f"| Macro F1 | {m['macro_f1']:.5f} |",
        f"| Top-3 accuracy | {m['top3_accuracy']:.5f} |",
        f"| ECE | {m['ece']:.5f} |",
        f"| MCE | {m['mce']:.5f} |",
        f"| Brier | {m['brier']:.5f} |",
        f"| Inference | {m['inference_ms_per_image']} ms/image |",
        "",
    ]

lines += [
    "## Notes",
    "",
    f"- Bootstrap resamples: {BOOTSTRAP} (image-level, percentile intervals).",
    "- The test split was scored once and was not used for model selection.",
    "- Validation was re-measured only to check the recorded score, not to choose anything.",
    "- Grad-CAM and field-domain (PlantDoc) results are separate, higher-phase artifacts.",
    "",
]
verification_path = OUT_ROOT / "P0.1_VERIFICATION.md"
verification_path.write_text("\n".join(lines), encoding="utf-8")
print("Wrote", verification_path)

zip_path = shutil.make_archive(str(OUT_ROOT), "zip", root_dir=OUT_ROOT)
print(f"\nDownload from the Kaggle Output panel: {zip_path}")
print("Contents:", sorted(p.name for p in OUT_ROOT.rglob('*') if p.is_file()))

## After this run

1. Download `p01_verification.zip`.
2. Unpack `validation/` and `test/` into `reports/model_comparison/locked_test_efficientnet_b0/` and keep `P0.1_VERIFICATION.md` next to them.
3. The test numbers in `metrics.json` become the only test numbers quoted in the guide form.
4. If validation macro-F1 does not reproduce the recorded 0.99369 within ~0.002, stop and investigate before quoting any test result - that would mean the checkpoint and the reported number disagree.

Still open after this notebook: P0.2 (ResNet18 retrain on the shared engine), P0.3 (Grad-CAM + calibration narrative), P0.4 (PlantDoc label map and first field evaluation), P0.5 (stale summary + survey title).